# Week 3: Data Cleaning and Preprocessing

## 1. Project Objective

The objective of this project is to inspect and clean the Adult Census Income dataset using Python and Pandas.

### Project Goals
- Identify missing values and explain how to handle them.
- Find and remove duplicate records when appropriate.
- Detect numeric outliers using the IQR method.
- Correct inconsistent categorical values.
- Encode categorical variables for downstream analysis.
- Compare the dataset before and after cleaning.

**Important:** The original dataset will remain unchanged. All cleaning operations will be performed on a working copy.

In [7]:
!pip install pandas


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
print(sys.executable)

c:\ProgramData\anaconda3\python.exe


In [2]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Libraries imported successfully!")

Pandas version: 3.0.3
NumPy version: 2.4.6
Libraries imported successfully!


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project folder
PROJECT_DIR = Path.cwd().parent

# Path to the original dataset
DATA_PATH = PROJECT_DIR / "data" / "raw" / "adult.data"

print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

Dataset path: c:\8th Sem\VK-Data Science\week-3\data\raw\adult.data
Dataset exists: True


In [4]:
# Column names from the Adult Census dataset documentation
columns = [
    "age", "workclass", "fnlwgt", "education",
    "education_num", "marital_status", "occupation",
    "relationship", "race", "sex", "capital_gain",
    "capital_loss", "hours_per_week", "native_country",
    "income"
]

# Load the original dataset without modifying it
df_original = pd.read_csv(
    DATA_PATH,
    header=None,
    names=columns,
    skipinitialspace=True,
    na_values="?"
)

# Create a working copy for later cleaning
df = df_original.copy()

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Dataset shape: (32561, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 2. Initial Data Quality Assessment

Before cleaning the dataset, we inspect its structure, data types, missing values, duplicate records, and sample rows. This establishes a baseline for measuring the impact of our cleaning decisions.

The original dataset is preserved in `df_original`, while `df` is our working copy.

In [5]:
# 1. Dataset dimensions
print("Dataset shape:", df.shape)

# 2. Column names and data types
print("\n--- Dataset Information ---")
df.info()

# 3. Missing values per column
print("\n--- Missing Values ---")
print(df.isnull().sum())

# 4. Duplicate rows
print("\n--- Duplicate Rows ---")
print("Number of duplicate rows:", df.duplicated().sum())

# 5. Numerical summary statistics
print("\n--- Numerical Summary ---")
display(df.describe())

# 6. Sample records
print("\n--- Sample Records ---")
display(df.head(10))

Dataset shape: (32561, 15)

--- Dataset Information ---
<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       30725 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education_num   32561 non-null  int64
 5   marital_status  32561 non-null  str  
 6   occupation      30718 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital_gain    32561 non-null  int64
 11  capital_loss    32561 non-null  int64
 12  hours_per_week  32561 non-null  int64
 13  native_country  31978 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 6.2 MB

--- Missing Values ---
age                  0
workclass         1836

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000



--- Sample Records ---


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
5,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K
6,49,Private,160187,9th,5,Married-spouse-absent,Other-service,Not-in-family,Black,Female,0,0,16,Jamaica,<=50K
7,52,Self-emp-not-inc,209642,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,45,United-States,>50K
8,31,Private,45781,Masters,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084,0,50,United-States,>50K
9,42,Private,159449,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,5178,0,40,United-States,>50K


In [6]:
# Missing values, from highest to lowest
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

display(missing_summary[missing_summary["Missing Count"] > 0]
        .sort_values("Missing Count", ascending=False))

print("Original dataset shape:", df_original.shape)
print("Duplicate rows:", df.duplicated().sum())

print("\nUnique categorical values:")
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(10))

,Missing Count,Missing Percentage
occupation,1843,5.66
workclass,1836,5.64
native_country,583,1.79


Original dataset shape: (32561, 15)
Duplicate rows: 24

Unique categorical values:

workclass:
workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
NaN                  1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

education:
education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
Name: count, dtype: int64

marital_status:
marital_status
Married-civ-spouse       14976
Never-married            10683
Divorced                  4443
Separated                 1025
Widowed                    993
Married-spouse-absent      418
Married-AF-spouse           23
Name: count, dtype: int64

occupation:
occupation
Prof-specialty       4140
Craft-repair         4099
Exec-managerial 

C:\Users\sayee\AppData\Local\Temp\ipykernel_14064\3511066467.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [7]:
# Original dataset quality report
quality_report = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2),
    "Unique Values": df.nunique(dropna=True)
})

display(quality_report)

print("Original shape:", df_original.shape)
print("Current working shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

print("\nNumeric columns:")
display(df.describe().T)

,Data Type,Missing Count,Missing Percentage,Unique Values
age,int64,0,0.00,73
workclass,str,1836,5.64,8
fnlwgt,int64,0,0.00,21648
education,str,0,0.00,16
education_num,int64,0,0.00,16
marital_status,str,0,0.00,7
occupation,str,1843,5.66,14
relationship,str,0,0.00,6
race,str,0,0.00,5
sex,str,0,0.00,2


Original shape: (32561, 15)
Current working shape: (32561, 15)
Duplicate rows: 24

Numeric columns:


,count,mean,std,min,25%,50%,75%,max
age,32561.0,38.581647,13.640433,17.0,28.0,37.0,48.0,90.0
fnlwgt,32561.0,189778.366512,105549.977697,12285.0,117827.0,178356.0,237051.0,1484705.0
education_num,32561.0,10.080679,2.572720,1.0,9.0,10.0,12.0,16.0
capital_gain,32561.0,1077.648844,7385.292085,0.0,0.0,0.0,0.0,99999.0
capital_loss,32561.0,87.303830,402.960219,0.0,0.0,0.0,0.0,4356.0
hours_per_week,32561.0,40.437456,12.347429,1.0,40.0,40.0,45.0,99.0


## 3. Initial Data Quality Findings

The original Adult Census Income dataset contains 32,561 rows and 15 columns.

The initial inspection identified the following issues:

1. **Missing values:** `workclass` contains 1,836 missing values, `occupation` contains 1,843, and `native_country` contains 583. These will be investigated individually before selecting an appropriate treatment.
2. **Duplicate records:** 24 duplicate rows were identified using Pandas' `duplicated()` method. These will be investigated and removed if they represent repeated records.
3. **Potential outliers:** Numeric summaries show extreme values in variables such as `capital_gain`, `capital_loss`, `fnlwgt`, and `hours_per_week`. These values require investigation before any removal decision.
4. **Categorical data:** Several columns contain categories that need inspection for missing-value markers, inconsistent formatting, and suitability for encoding.

No cleaning has been performed yet. These findings establish the baseline for the before-and-after comparison.

In [8]:
# Record the shape before removing duplicates
shape_before_duplicates = df.shape
duplicates_before = df.duplicated().sum()

# Remove exact duplicate rows from the working copy
df = df.drop_duplicates().copy()

# Verify the result
print("Shape before:", shape_before_duplicates)
print("Duplicate rows found:", duplicates_before)
print("Shape after:", df.shape)
print("Remaining duplicate rows:", df.duplicated().sum())

Shape before: (32561, 15)
Duplicate rows found: 24
Shape after: (32537, 15)
Remaining duplicate rows: 0


In [9]:
# Inspect missing values after duplicate removal
missing_before = df.isnull().sum()

print("Missing values after duplicate removal:")
display(
    missing_before[missing_before > 0]
    .rename("Missing Count")
    .to_frame()
)

print("Current dataset shape:", df.shape)

Missing values after duplicate removal:


,Missing Count
workclass,1836
occupation,1843
native_country,582


Current dataset shape: (32537, 15)


## 4. Handling Missing Values

After removing 24 duplicate rows, the dataset contains 32,537 records.

The missing-value treatment is selected separately for each affected column:

- **`workclass`:** Replace missing values with `"Unknown"` because the employment class cannot be reliably inferred from other columns.
- **`occupation`:** Replace missing values with `"Unknown"` because assigning the most frequent occupation could misrepresent an individual's occupation.
- **`native_country`:** Replace missing values with `"Unknown"` because the person's country of origin is not known. Inferring it could introduce unsupported assumptions.

No rows will be dropped solely because these categorical values are missing. This preserves the available information while making missingness explicit.

In [10]:
# Record missing-value counts before treatment
missing_before_treatment = df.isnull().sum()

# Replace missing categorical values with an explicit category
missing_columns = [
    "workclass",
    "occupation",
    "native_country"
]

for col in missing_columns:
    df[col] = df[col].fillna("Unknown")

# Verify the treatment
print("Missing values before treatment:")
display(missing_before_treatment[missing_before_treatment > 0])

print("\nMissing values after treatment:")
display(df.isnull().sum())

print("\nDataset shape after missing-value treatment:", df.shape)

Missing values before treatment:


workclass         1836
occupation        1843
native_country     582
dtype: int64


Missing values after treatment:


age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64


Dataset shape after missing-value treatment: (32537, 15)


## 5. Outlier Detection Using the IQR Method

The Interquartile Range (IQR) method identifies unusually low or high numeric values.

IQR = Q3 − Q1

- Q1 is the 25th percentile.
- Q3 is the 75th percentile.
- Lower bound = Q1 − 1.5 × IQR.
- Upper bound = Q3 + 1.5 × IQR.

Values outside these bounds are flagged as potential outliers. They are not automatically errors, so each variable must be assessed based on its meaning and the observed values.

For this dataset, we will investigate `age`, `fnlwgt`, `capital_gain`, `capital_loss`, and `hours_per_week`. We will document our decisions before modifying any values.

In [11]:
# Columns to investigate for potential outliers
numeric_columns = [
    "age",
    "fnlwgt",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

outlier_results = []

for col in numeric_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)

    outlier_results.append({
        "Column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outlier_mask.sum()),
        "Minimum": df[col].min(),
        "Maximum": df[col].max()
    })

outlier_report = pd.DataFrame(outlier_results)
display(outlier_report.round(2))

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count,Minimum,Maximum
0,age,28.0,48.0,20.0,-2.0,78.0,142,17,90
1,fnlwgt,117827.0,236993.0,119166.0,-60922.0,415742.0,993,12285,1484705
2,capital_gain,0.0,0.0,0.0,0.0,0.0,2712,0,99999
3,capital_loss,0.0,0.0,0.0,0.0,0.0,1519,0,4356
4,hours_per_week,40.0,45.0,5.0,32.5,52.5,9002,1,99


### IQR Outlier Analysis and Decision

The IQR method identified potential outliers in five numerical columns: `age`, `fnlwgt`, `capital_gain`, `capital_loss`, and `hours_per_week`.

The `capital_gain` and `capital_loss` columns have an IQR of zero because at least 75% of their values are zero. Therefore, the IQR method flags many positive values as potential outliers. These values may represent genuine financial activity.

The `fnlwgt` column contains sampling weights, and extreme values may be valid. Similarly, unusual ages and working hours are not necessarily data-entry errors.

**Decision:** No rows or values will be removed solely because they are flagged by the IQR method. The detected values will be retained because there is insufficient evidence that they are incorrect. This avoids unnecessary information loss during preprocessing.

In [12]:
categorical_columns = df.select_dtypes(
    include=["object"]
).columns

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(20))


--- workclass ---
workclass
Private             22673
Self-emp-not-inc     2540
Local-gov            2093
Unknown              1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

--- education ---
education
HS-grad         10494
Some-college     7282
Bachelors        5353
Masters          1722
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           645
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           332
1st-4th           166
Preschool          50
Name: count, dtype: int64

--- marital_status ---
marital_status
Married-civ-spouse       14970
Never-married            10667
Divorced                  4441
Separated                 1025
Widowed                    993
Married-spouse-absent      418
Married-AF-spouse           23
Name: count, dtype: int64

--- occupation ---
oc

C:\Users\sayee\AppData\Local\Temp\ipykernel_14064\1047149483.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


In [13]:
# Identify categorical columns
categorical_columns = df.select_dtypes(
    include=["object"]
).columns

# Remove leading and trailing spaces
for col in categorical_columns:
    df[col] = df[col].str.strip()

print("Categorical values standardized successfully!")

# Verify the results
for col in categorical_columns:
    print(f"{col}: {df[col].nunique()} unique values")

Categorical values standardized successfully!
workclass: 9 unique values
education: 16 unique values
marital_status: 7 unique values
occupation: 15 unique values
relationship: 6 unique values
race: 5 unique values
sex: 2 unique values
native_country: 42 unique values
income: 2 unique values


C:\Users\sayee\AppData\Local\Temp\ipykernel_14064\722720178.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


### Encoding Categorical Variables

Categorical variables contain text values that need to be converted into numerical representations for many machine learning algorithms.

In this dataset:
- **One-hot encoding** will be used for nominal categorical features, such as `workclass`, `marital_status`, `occupation`, `relationship`, `race`, `sex`, and `native_country`.
- `education` will be excluded from the feature encoding because `education_num` already represents the education level numerically.
- The `income` column will be converted into a binary target label: `<=50K` becomes `0`, and `>50K` becomes `1`.

One-hot encoding avoids assigning artificial numerical rankings to unordered categories.


In [14]:
# Create a separate copy for encoding
df_encoded = df.copy()

# Convert the target column into binary values
df_encoded["income"] = df_encoded["income"].map({
    "<=50K": 0,
    ">50K": 1
})

# Remove the redundant education text column
df_encoded = df_encoded.drop(columns=["education"])

# Identify categorical feature columns
categorical_features = df_encoded.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical columns to encode:")
print(categorical_features)

# Apply one-hot encoding
df_encoded = pd.get_dummies(
    df_encoded,
    columns=categorical_features,
    dtype=int
)

print("\nOriginal cleaned dataset shape:", df.shape)
print("Encoded dataset shape:", df_encoded.shape)

print("\nFirst five rows of encoded data:")
display(df_encoded.head())

Categorical columns to encode:
['workclass', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']

Original cleaned dataset shape: (32537, 15)
Encoded dataset shape: (32537, 93)

First five rows of encoded data:


C:\Users\sayee\AppData\Local\Temp\ipykernel_14064\2383232938.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df_encoded.select_dtypes(


,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week,income,workclass_Federal-gov,workclass_Local-gov,workclass_Never-worked,...,native_country_Puerto-Rico,native_country_Scotland,native_country_South,native_country_Taiwan,native_country_Thailand,native_country_Trinadad&Tobago,native_country_United-States,native_country_Unknown,native_country_Vietnam,native_country_Yugoslavia
0,39,77516,13,2174,0,40,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,50,83311,13,0,0,13,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,38,215646,9,0,0,40,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,53,234721,7,0,0,40,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,28,338409,13,0,0,40,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
categorical_features = df_encoded.select_dtypes(
    include=["object"]
).columns.tolist()

In [16]:
categorical_features = df_encoded.select_dtypes(
    include=["object", "str"]
).columns.tolist()

In [17]:
print("Missing values:", df_encoded.isnull().sum().sum())
print("Duplicate rows:", df_encoded.duplicated().sum())

print("\nIncome target distribution:")
print(df_encoded["income"].value_counts().sort_index())

print("\nAll columns are numerical:",
      df_encoded.select_dtypes(exclude="number").empty)

Missing values: 0
Duplicate rows: 0

Income target distribution:
income
0    24698
1     7839
Name: count, dtype: int64

All columns are numerical: True


### Before-and-After Data Quality Comparison

This section compares the original dataset, the cleaned dataset, and the encoded dataset to evaluate the effectiveness of the preprocessing steps.

The comparison includes dataset dimensions, missing values, duplicate rows, and data types. The original dataset is retained as a baseline so that the impact of cleaning and encoding can be measured.

The cleaning process removes duplicate records, replaces missing categorical values with `Unknown`, standardizes categorical text, and converts categorical features into numerical representations.


In [18]:
# Create a data quality comparison report
comparison_report = pd.DataFrame({
    "Metric": [
        "Number of rows",
        "Number of columns",
        "Missing values",
        "Duplicate rows"
    ],
    "Original Dataset": [
        df_original.shape[0],
        df_original.shape[1],
        int(df_original.isnull().sum().sum()),
        int(df_original.duplicated().sum())
    ],
    "Cleaned Dataset": [
        df.shape[0],
        df.shape[1],
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum())
    ],
    "Encoded Dataset": [
        df_encoded.shape[0],
        df_encoded.shape[1],
        int(df_encoded.isnull().sum().sum()),
        int(df_encoded.duplicated().sum())
    ]
})

display(comparison_report)

,Metric,Original Dataset,Cleaned Dataset,Encoded Dataset
0,Number of rows,32561,32537,32537
1,Number of columns,15,15,93
2,Missing values,4262,0,0
3,Duplicate rows,24,0,0


In [19]:
# Create the processed data directory if needed
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset
cleaned_path = PROCESSED_DIR / "adult_cleaned.csv"
df.to_csv(cleaned_path, index=False)

# Save the encoded dataset
encoded_path = PROCESSED_DIR / "adult_encoded.csv"
df_encoded.to_csv(encoded_path, index=False)

print("Datasets saved successfully!")
print("Cleaned dataset:", cleaned_path)
print("Encoded dataset:", encoded_path)

print("\nSaved file sizes:")
print("Cleaned:", round(cleaned_path.stat().st_size / (1024 * 1024), 2), "MB")
print("Encoded:", round(encoded_path.stat().st_size / (1024 * 1024), 2), "MB")

Datasets saved successfully!
Cleaned dataset: c:\8th Sem\VK-Data Science\week-3\data\processed\adult_cleaned.csv
Encoded dataset: c:\8th Sem\VK-Data Science\week-3\data\processed\adult_encoded.csv

Saved file sizes:
Cleaned: 3.41 MB
Encoded: 6.05 MB


## Conclusion

In this project, the Adult Census Income dataset was cleaned and preprocessed to improve its quality and prepare it for machine learning.

The following tasks were completed:

1. Loaded and inspected the dataset containing 32,561 rows and 15 columns.
2. Identified missing values and 24 duplicate rows.
3. Removed duplicate records, resulting in 32,537 rows.
4. Handled missing categorical values by replacing them with `Unknown`.
5. Standardized categorical values by removing unnecessary spaces.
6. Used the IQR method to identify potential outliers and retained plausible values rather than removing them automatically.
7. Applied one-hot encoding to categorical features and converted the income target into binary labels.
8. Verified that the encoded dataset contained no missing values or duplicate rows and that all columns were numerical.
9. Saved the cleaned and encoded datasets as CSV files.

The final encoded dataset contains 32,537 rows and 93 columns. The preprocessing workflow demonstrates how data quality issues can be identified, handled, and documented before machine learning.

### Output Files

- `data/processed/adult_cleaned.csv`
- `data/processed/adult_encoded.csv`

**Outcome:** The dataset is prepared for further exploratory data analysis and machine learning experiments.
